# Vectors Stores and Retrievers

Vector stores and retrievers are important components in LangChain for retrieving relevant information from vector databases and other data sources.

They enable LLM applications to retrieve relevant data and provide it as context to language models, making them especially useful for Retrieval-Augmented Generation (RAG) applications.

In this notebook, we will work with:

- Documents: Creating and managing documents with metadata.
- Vector Stores: Storing document embeddings and performing similarity searches.
- Retrievers: Retrieving relevant documents from a data source.
- RAG: Using retrieved documents as context for generating LLM responses.

## Documents

LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has 2 attributes:

- page_content: a string representing the content
- metadata: a dict containing arbitrary metadata. The metadata attribute can capture information about the source of the document, its relationship to other documents and other information.

Note that an individual Document object often represents a chunk of a larger document.

### Creating Documents with Metadata

In [19]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source":"fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source":"bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that needs plenty of space to hop around.",
        metadata={"source":"mammal-pets-doc"},
    ),
]

### Inspecting the Documents

In [20]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that needs plenty of space to hop around.')]

## Setting Up the LLM and Embeddings

### Initializing the Groq Chat Model

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
llm=ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002541D715FF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002541D716200>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Using HuggingFace Embeddings

In [22]:
### We use HuggingFace embedding techniques

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4784.15it/s]


## Vector Stores

### Creating a Chroma Vector Store

In [23]:
### Vector Stores

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore

### Performing Similarity Search

In [24]:
vectorstore.similarity_search("cat")

[Document(id='0022be8c-fa22-40ff-b567-7c45291638e4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='b2c78ef2-c2b2-4608-b073-965691669b33', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='0b13bc4c-19d8-48a6-8696-0d029d0de2e8', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='43c20ea2-6260-40a3-a554-fb029c100def', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

### Performing Asynchronous Similarity Search

In [25]:
### Async Query

await vectorstore.asimilarity_search("cat")

[Document(id='0022be8c-fa22-40ff-b567-7c45291638e4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='b2c78ef2-c2b2-4608-b073-965691669b33', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='0b13bc4c-19d8-48a6-8696-0d029d0de2e8', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='43c20ea2-6260-40a3-a554-fb029c100def', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

### Similarity Search with Scores

In [26]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='0022be8c-fa22-40ff-b567-7c45291638e4', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351057410240173),
 (Document(id='b2c78ef2-c2b2-4608-b073-965691669b33', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351057410240173),
 (Document(id='0b13bc4c-19d8-48a6-8696-0d029d0de2e8', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='43c20ea2-6260-40a3-a554-fb029c100def', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956)]

## Retrievers

LangChain VectorStore objects do not subclass Runnable and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method.

### Creating a Retriever with RunnableLambda

In [27]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

### Retrieving Top-K Documents
retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)    # select the top result : k=1
retriever.batch(["cat", "dog"])

[[Document(id='b2c78ef2-c2b2-4608-b073-965691669b33', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='0b13bc4c-19d8-48a6-8696-0d029d0de2e8', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specially a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call and how to parameterize them. For instance, we can replicate the above with the following:

### Creating a Vector Store Retriever

In [28]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat", "dog"])

[[Document(id='b2c78ef2-c2b2-4608-b073-965691669b33', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='0b13bc4c-19d8-48a6-8696-0d029d0de2e8', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

## RAG - Retrieval Augmented Generation

In [29]:
### RAG - Retrievel Augmented Generation

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer this question using the porvided context only.
{question}

Context:
{context}
"""

### Creating the RAG Prompt

In [30]:
prompt=ChatPromptTemplate.from_messages([("human", message)])

### Building the RAG Chain

In [31]:
rag_chain={"context":retriever, "question":RunnablePassthrough()}|prompt|llm

### Generating Responses with Retrieved Context

In [32]:
response=rag_chain.invoke("Tell me about docs")
response.content

'Based on the only document provided:\n\n- **Source:** The document comes from *bird-pets-doc*.\n- **Content:** It states that **parrots are intelligent birds capable of mimicking human speech**.'